In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

In [ ]:
import os
os.getcwd()

# Import File CSV ke DataFrame

In [ ]:
path = r"C:\Users\Nur Aini Mahfud\Downloads\Salinan Salinan Online Retail Data.csv"
df = pd.read_csv(path)
df

In [ ]:
df.info()

# Data Cleansing

In [ ]:
df_clean = df.copy()
df_clean['date'] = pd.to_datetime(df_clean['order_date']).dt.date.astype('datetime64[ns]')
df_clean = df_clean[~df_clean['customer_id'].isna()]
df_clean = df_clean[~df_clean['product_name'].isna()]
df_clean['product_name'] = df_clean['product_name'].str.lower()
df_clean = df_clean[(~df_clean['product_code'].str.lower().str.contains('test')) |
                    (~df_clean['product_name'].str.contains('test '))]
df_clean = df_clean[df_clean['order_id'].str[:1]!='C']
df_clean['quantity'] = df_clean['quantity'].abs()
df_clean = df_clean[df_clean['price']>0]
df_clean['amount'] = df_clean['quantity'] * df_clean['price']

In [ ]:
most_freq_product_name = df_clean.groupby(['product_code','product_name'], as_index=False).agg(order_cnt=('order_id','nunique')).sort_values(['product_code','order_cnt'], ascending=[True,False])
most_freq_product_name['rank'] = most_freq_product_name.groupby('product_code')['order_cnt'].rank(method='first', ascending=False)
most_freq_product_name = most_freq_product_name[most_freq_product_name['rank']==1].drop(columns=['order_cnt','rank'])
df_clean = df_clean.merge(most_freq_product_name.rename(columns={'product_name':'most_freq_product_name'}), how='left', on='product_code')
df_clean['product_name'] = df_clean['most_freq_product_name']
df_clean = df_clean.drop(columns='most_freq_product_name')
df_clean['customer_id'] = df_clean['customer_id'].astype(str)

from scipy import stats
df_clean = df_clean[(np.abs(stats.zscore(df_clean[['quantity','amount']]))<3).all(axis=1)]
df_clean = df_clean.reset_index(drop=True)
df_clean

In [ ]:
df_clean.info()

# Preparing Data Basket

**1.Membuat DataFrame Basket**

In [ ]:
basket = pd.pivot_table(df_clean, index='order_id', columns='product_name', values='product_code', aggfunc='nunique', fill_value=0)
basket

In [ ]:
basket.info()

**2.Encode DataFrame basket dengan nilai True untuk semua nilai di atas 0 dan False untuk semua nilai 0**

In [ ]:
def encode(x):
    if x==0:
        return False 
    if x>0:
        return True

basket_encode = basket_encode.astype('int8')
basket

In [ ]:
basket_encode.info()

**3. Mengambil Transaksi banyaknya produk unik lebih dari 1 saja**

In [ ]:
basket_filter = basket_encode[(basket_encode>0).sum(axis=1)>1]
basket_filter

In [ ]:
basket_filter.info()

# Mengaplikasikan Apriori Algorithm

**Membuat List Frequent Itemset (Kumpulan Produk Yang Sering Dibeli)**

In [ ]:
pip install mlxtend

In [ ]:
from mlxtend.frequent_patterns import apriori

basket_filter = basket_filter.astype('int8')

frequent_itemset = apriori(basket_filter, min_support=.01, use_colnames=True, low_memory=True).sort_values('support', ascending=False).reset_index(drop=True)
frequent_itemset['product_cnt'] = frequent_itemset['itemsets'].apply(lambda x: len(x))
frequent_itemset

# Menghitung NIlai Support, Confidence, Lift dari Setiap Pasangan Produk Yang Mungkin

In [ ]:
from mlxtend.frequent_patterns import association_rules 

product_association = association_rules(frequent_itemset, metric='confidence', min_threshold=.7).sort_values(['support', 'confidence'], ascending=[False,False]).reset_index(drop=True)
product_association

In [ ]:
product_association_simple = product_association[['antecedents','consequents','antecedent support', 'consequent support', 'support','confidence','lift', 'leverage', 'conviction']]
product_association_simple

# INSIGHT #